Для Fine Tuning мы будем использовать ту предобработку датасета, что была для промтинга. Задача Fine Tuning это побить бейзлайн, который был выбит на промтинге с лёгкой моделью qwen и базовым промтом. В ходе Fine Tuning мы не будем отпиливать голову модели, верхние слои или переобучать все веса. Последнее особенно долго и неудобно. Поэтому воспользуемся LoRA адаптером.

## Универсальный код для загрузки датасета

In [ ]:
import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
print(f"Устройство: {torch.cuda.get_device_name(0)}")
print(f"Память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import pandas as pd
import kagglehub
import os

path = kagglehub.dataset_download('xuanhuynh233/ielts-dataset')

train_df = pd.read_csv(os.path.join(path, 'train_final.csv'))
test_df  = pd.read_csv(os.path.join(path, 'test_final.csv'))

print(f"Train: {len(train_df)} строк")
print(f"Test:  {len(test_df)} строк")

def clean_df(df, name):
    initial_len = len(df)
    df = df.dropna(subset=['essay', 'band'])
    df = df[pd.to_numeric(df['band'], errors='coerce').notna()]
    df['band'] = df['band'].astype(float)
    print(f"[{name}] Удалено {initial_len - len(df)} строк → осталось {len(df)}")
    return df

train_df = clean_df(train_df, 'TRAIN')
test_df  = clean_df(test_df,  'TEST')

## Загружаем qwen

In [ ]:
!pip uninstall -y bitsandbytes
!pip install -q bitsandbytes==0.46.1

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Конфигурация 4-bit квантизации - чтобы быстро скачать и чтобы быстро работало (для пробы пера)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_name = "Qwen/Qwen2.5-3B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = 'right' # Добавляем, чтобы нормально модель нормально обучалась

print(f'Параметров: {model.num_parameters():,}')

## Промт

Не смотря на то, что мы обучаем модель, она не знает что мы ей даём на входе. Для обучения всё равно необходимо, чтобы модель понимала задачу, которую будет решать. Но здесь в отличие от чистого промтинга, не нужно указывать примеры, так как модель и так будет на них обучаться. Будем использовать instruction + zero-shot.

In [ ]:
# сразу все импорты
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr
import re
from tqdm import tqdm
import torch

Будем учитывать само задание, чтобы модель более корректно оценивала Task Response

In [ ]:
# промт на вход модели
def prompt(essay, task_prompt):
    return f"""You are an expert IELTS Writing Task 2 examiner. Grade essays using IELTS band descriptors (0-9, step 0.5).

IELTS CRITERIA (each 0-9):
1. TR (Task Response): How well the essay addresses the task, presents ideas, and supports arguments
2. CC (Coherence & Cohesion): Organization, paragraph structure, and linking devices
3. LR (Lexical Resource): Vocabulary range, accuracy, and appropriateness
4. GRA (Grammatical Range & Accuracy): Grammar variety, accuracy, and complexity

IMPORTANT:
- Use only 0.5 increments (e.g., 6.0, 6.5, 7.0)
- Overall Band = Average of all 4 criteria
- Provide realistic scores based on IELTS standards

Task: {task_prompt}

Essay: {essay}

Provide your evaluation in this EXACT format:
TR_Band: X.X
CC_Band: X.X
LR_Band: X.X
GRA_Band: X.X
Overall_Band: X.X
Explanation: [Brief justification for each criterion]"""

In [ ]:
# правильный ответ
def answer(row):
    return f"""TR_Band: {row['TR_Band']}
CC_Band: {row['CC_Band']}
LR_Band: {row['LR_Band']}
GRA_Band: {row['GRA_Band']}
Overall_Band: {row['Overall_Band']}
Explanation: {row['General_Feedback']}"""

Во время обучения модель до того, как увидеть настоящий ответ, предсказывает свой и потом считает лосс в сравнении с настоящим.

In [ ]:
# преобразуем данные в нужный формат для модели
def df_to_messages(df):
    records = []
    for _, row in df.iterrows():
        records.append({
            'messages': [
                {'role': 'user', 'content': prompt(row['essay'], row['prompt'])},
                {'role': 'assistant', 'content': answer(row)},
            ]
        })
    return records

## LoRA

In [ ]:
!pip install -q peft trl

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

lora_config = LoraConfig(
    r=16, # ранг матриц обновления
    lora_alpha=32, # скорость обучения
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention
        "gate_proj", "up_proj", "down_proj", # MLP
    ],
    lora_dropout=0.05, # регуляризация
    bias="none",
    task_type="CAUSAL_LM", # тип задачи
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
# model.gradient_checkpointing_enable()
model.print_trainable_parameters()

https://huggingface.co/docs/peft/package_reference/lora

Обучаем 0,96% весов модели

## Токенизация датасета

In [ ]:
#!pip install -q pyarrow --upgrade
#перезапустить сеанс

In [ ]:
from datasets import Dataset

max_length = 1024

def tokenize_dataset(records):
    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for rec in records:
        # строим полный чат
        text = tokenizer.apply_chat_template(
            rec['messages'],
            tokenize=False,
            add_generation_prompt=False
        )

        # токенизация
        tokenized = tokenizer(
            text,
            max_length=max_length,
            padding='max_length',
            truncation=True,
        )

        input_ids = tokenized['input_ids']
        attention_mask = tokenized['attention_mask']

        assistant_start = text.find('<|im_start|>assistant')
        prompt_part = tokenizer(text[:assistant_start], add_special_tokens=False)['input_ids']
        prompt_len = len(prompt_part)

        labels = input_ids.copy()
        labels[:prompt_len] = [-100] * prompt_len # заменяем на -100 всё, кроме ответа

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    return Dataset.from_dict({
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list
    })

In [ ]:
# train_df = train_df.sample(10, random_state=42) #Проверяем на мелком

In [ ]:
train_records = df_to_messages(train_df)
test_records = df_to_messages(test_df)

Нужно посмотреть, сколько наблюдений не влезло в 1024 токена. Посмотрим на тренировочной выборке, так как она больше.

In [ ]:
overflow_count = 0
for rec in train_records:
    text = tokenizer.apply_chat_template(
        rec['messages'], tokenize=False, add_generation_prompt=False
    )
    tokens = tokenizer(text, add_special_tokens=False)['input_ids']
    if len(tokens) > 1024:
        overflow_count += 1

print(f"Примеров длиннее 1024 токенов: {overflow_count} из {len(train_records)}")
print(f"Это {overflow_count / len(train_records) * 100:.2f}% датасета")

Не влезает 0,5% датасета. Чтобы уменьшить риск ошибок при обучении их лучше выкинуть до этапа токенизации датасета.

In [ ]:
def filter_long_records(records, max_length):
    filtered = []
    for rec in records:
        text = tokenizer.apply_chat_template(
            rec['messages'], tokenize=False, add_generation_prompt=False
        )
        tokens = tokenizer(text, add_special_tokens=False)['input_ids']
        if len(tokens) <= max_length:
            filtered.append(rec)
    print(f"Отфильтровано {len(records) - len(filtered)} примеров, осталось {len(filtered)}")
    return filtered

In [ ]:
train_records = filter_long_records(train_records, max_length)
test_records  = filter_long_records(test_records, max_length)

In [ ]:
train_dataset = tokenize_dataset(train_records)
test_dataset = tokenize_dataset(test_records)

train_dataset.set_format('torch')
test_dataset.set_format('torch')

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")

# проверяем что маскировка работает
sample_labels = train_dataset[0]['labels']
n_masked = (sample_labels == -100).sum().item()
n_real = (sample_labels != -100).sum().item()
print(f"Замаскировано {n_masked} токенов (промт), обучаемых {n_real} токенов (ответ)")

## Обучение

In [ ]:
from transformers import TrainingArguments, Trainer, default_data_collator, TrainerCallback

OUTPUT_DIR = "./qwen-ielts-lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    fp16=True,

    eval_strategy="epoch",
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=False,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=8,
    #metric_for_best_model="eval_loss",
    #greater_is_better=False,

    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=default_data_collator
)

print("Начинаем обучение...")
trainer.train()

Обходим ограничение на сессию в 12 часов

In [ ]:
import os
print(os.listdir('/kaggle/working/qwen-ielts-lora'))

In [ ]:
from huggingface_hub import login

login(token=TOKEN_HUG)

model.push_to_hub("salarion-witch/qwen-ielts-lora")
tokenizer.push_to_hub("salarion-witch/qwen-ielts-lora")
print("Сохранено на HuggingFace!")

шагов за эпоху = размер датасета / (batch_size × gradient_accumulation_steps)

шагов за эпоху = 1000 / (2 × 8) = 62,5

Шагов оптимизатора за эпоху всего примерно 62, а logging_steps по дефолту равны 500. Получается лог срабатывает один раз на 500-м шаге, но Training показывает его только если в интервале был хотя бы один полный цикл логирования. При такой конфигурации он просто не успевает записать. Поэтому в нашей ситуации в Traing Loss не проливается Loss.

In [ ]:
model.save_pretrained(OUTPUT_DIR + '/lora_adapter')
tokenizer.save_pretrained(OUTPUT_DIR + '/lora_adapter')

## Предсказание и метрики

In [ ]:
test_sample = test_df.sample(50, random_state=42).copy()

In [ ]:
# функция генерации оценки
def grade_essay(essay, task_prompt, max_tokens=300):
    messages = [{'role': 'user', 'content': prompt(essay, task_prompt)}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            repetition_penalty=1.1,
        )

    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


# функция извлечения оценок из ответа модели
def extract_scores(response):
    patterns = {
        'TR': r'TR[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'CC': r'CC[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'LR': r'LR[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'GRA': r'GRA[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'Overall': r'Overall[_\s]*Band:\s*(\d+(?:\.\d)?)',
    }
    scores = {}
    for criterion, pattern in patterns.items():
        match = re.search(pattern, response, re.IGNORECASE)
        if match:
            score = float(match.group(1))
            scores[criterion] = min(max(round(score * 2) / 2, 0), 9)
        else:
            scores[criterion] = None

    # если Overall не распарсился, считаем как среднее
    if scores['Overall'] is None and all(scores[k] is not None for k in ['TR', 'CC', 'LR', 'GRA']):
        avg = np.mean([scores['TR'], scores['CC'], scores['LR'], scores['GRA']])
        scores['Overall'] = round(avg * 2) / 2

    return scores


# запускаем предсказания
predictions = {'TR': [], 'CC': [], 'LR': [], 'GRA': [], 'Overall': []}

print(f"Оценка {len(test_sample)} IELTS эссе...")
for idx, row in tqdm(test_sample.iterrows(), total=len(test_sample)):
    try:
        response = grade_essay(row['essay'], row['prompt'])
        scores = extract_scores(response)
        for criterion in predictions:
            predictions[criterion].append(scores[criterion])
    except Exception as e:
        print(f"\nОшибка на эссе {idx}: {e}")
        for criterion in predictions:
            predictions[criterion].append(None)

# добавляем предсказания в датафрейм
for criterion in predictions:
    test_sample[f'pred_{criterion}'] = predictions[criterion]

print("\nОценка завершена!")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, cohen_kappa_score
from scipy.stats import spearmanr

def calculate_metrics(actual, predicted):
    mask = ~(pd.isna(actual) | pd.isna(predicted))
    actual = np.array(actual)[mask]
    predicted = np.array(predicted)[mask]

    if len(actual) == 0:
        return None

    correlation, _ = spearmanr(actual, predicted)
    qwk = cohen_kappa_score(
        (actual * 2).astype(int),
        (predicted * 2).astype(int),
        weights='quadratic'
    )

    return {
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
        'MBE': np.mean(predicted - actual),
        'Correlation': correlation,
        'QWK': qwk,
    }


criteria_mapping = {
    'TR': 'TR_Band',
    'CC': 'CC_Band',
    'LR': 'LR_Band',
    'GRA': 'GRA_Band',
    'Overall': 'Overall_Band',
}

all_metrics = {}
for criterion, actual_col in criteria_mapping.items():
    metrics = calculate_metrics(test_sample[actual_col], test_sample[f'pred_{criterion}'])
    if metrics:
        all_metrics[criterion] = metrics

metrics_df = pd.DataFrame(all_metrics).T.round(3)
print(metrics_df[['MAE', 'RMSE', 'MBE', 'Correlation', 'QWK']])

In [ ]:
print(test_sample['pred_LR'].value_counts())
print(test_sample['pred_GRA'].value_counts())

In [ ]:
for col in ['pred_TR', 'pred_CC', 'pred_LR', 'pred_GRA', 'pred_Overall']:
    print(f"{col}: {test_sample[col].value_counts().to_dict()}")

In [ ]:
# сохраняем предсказания и метрики
test_sample.to_csv('/kaggle/working/ielts_predictions.csv', index=False)
metrics_df.to_csv('/kaggle/working/ielts_metrics.csv')
print("Предсказания и метрики сохранены!")